# 02 -- Feature Engineering

Turns `data/orders.csv` into a model-ready table for the return-risk
scorer, and produces a **chronological** train/test split (not random).

## Why chronological, not random

A random split can leak information: a customer's orders from later in
the year could end up in "train" alongside earlier orders from the same
customer in "test", letting the model partially learn a customer's future
via their other rows. A chronological split -- train on the first ~75% of
the timeline, test on the last ~25% -- mimics how the model would actually
be deployed (trained on the past, scored on the future) and gives an
honest estimate of out-of-time performance.

## Modeling population

Only **returned** orders are used. The question this model answers is
*"given a return is happening, how likely is it abusive?"* -- not *"will
this order be returned?"*, which is a related but separate problem.

## Feature list deliberately excludes

- `order_id`, `customer_id`, `order_date` -- identifiers / leakage risk
- device/address/payment fingerprints -- reserved for the abuse-ring
  sentinel (notebook 04), a separate model with a separate job
- `_behavior_type` -- the audit-only ground-truth generator from notebook
  01. Using it as a feature would be cheating: it's essentially the label
  generator, not something a real deployment would ever observe.

In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

Working directory: /home/claude/riskguard


In [2]:
import pandas as pd

df = pd.read_csv("data/orders.csv", parse_dates=["order_date"])
print(f"Total orders: {len(df)}")

returns = df[df["is_returned"] == 1].copy()
returns = returns.sort_values("order_date").reset_index(drop=True)
print(f"Returned orders (our modeling population): {len(returns)}")

Total orders: 50108
Returned orders (our modeling population): 10305


## No-leakage check: rolling features must only see the past

`hist_return_rate_before`, `hist_abusive_return_rate_before`, and
`hist_chargebacks_before` are computed per customer using only orders
strictly *before* the current one, in chronological order. This is the
single easiest place to accidentally leak the future into a feature --
worth a direct sanity check rather than trusting the code silently.

In [3]:
# Sanity check: for any customer with multiple orders, hist_orders_before
# should be strictly increasing over time (0, 1, 2, ... for that customer's
# successive orders) -- if it ever repeats or goes backwards, that's a sign
# the rolling computation isn't respecting chronological order.
sample_customer = df[df["customer_id"].map(df["customer_id"].value_counts()) >= 4]["customer_id"].iloc[0]
sample_orders = df[df["customer_id"] == sample_customer].sort_values("order_date")
print(f"Sample customer {sample_customer}'s orders over time:")
sample_orders[["order_date", "hist_orders_before", "hist_return_rate_before", "is_returned"]]

Sample customer CUST100000's orders over time:


,order_date,hist_orders_before,hist_return_rate_before,is_returned
0,2025-02-13,0,0.0000,0
1,2025-03-30,1,0.0000,0
2,2025-04-17,2,0.0000,1
3,2025-04-26,3,0.3333,1
4,2025-05-29,4,0.5000,1
5,2025-05-30,5,0.6000,0
6,2025-08-28,6,0.5000,0
7,2025-09-11,7,0.4286,0
8,2025-10-25,8,0.3750,0
9,2025-10-26,9,0.3333,0


In [4]:
NUMERIC_FEATURES = [
    "order_value", "delivery_days", "order_hour", "is_weekend",
    "account_age_days_at_order", "hist_orders_before",
    "hist_return_rate_before", "hist_abusive_return_rate_before",
    "hist_chargebacks_before", "price_vs_category_avg", "days_to_return",
]
CATEGORICAL_FEATURES = ["category", "payment_method", "return_reason"]
TARGET = "is_abusive_return"

encoded = pd.get_dummies(returns[CATEGORICAL_FEATURES], prefix=CATEGORICAL_FEATURES)
feature_df = pd.concat(
    [returns[["order_id", "customer_id", "order_date"]],
     returns[NUMERIC_FEATURES], encoded, returns[[TARGET]]],
    axis=1,
)
feature_cols = NUMERIC_FEATURES + list(encoded.columns)
print(f"Feature count: {len(feature_cols)}")
feature_cols

Feature count: 28


['order_value',
 'delivery_days',
 'order_hour',
 'is_weekend',
 'account_age_days_at_order',
 'hist_orders_before',
 'hist_return_rate_before',
 'hist_abusive_return_rate_before',
 'hist_chargebacks_before',
 'price_vs_category_avg',
 'days_to_return',
 'category_accessories',
 'category_apparel',
 'category_beauty',
 'category_electronics',
 'category_footwear',
 'category_home',
 'category_mobile',
 'payment_method_COD',
 'payment_method_UPI',
 'payment_method_card',
 'payment_method_wallet',
 'return_reason_changed_mind',
 'return_reason_damaged',
 'return_reason_no_longer_needed',
 'return_reason_not_as_described',
 'return_reason_size_issue',
 'return_reason_wrong_item']

In [5]:
cutoff = feature_df["order_date"].quantile(0.75, interpolation="nearest")
train = feature_df[feature_df["order_date"] <= cutoff].copy()
test = feature_df[feature_df["order_date"] > cutoff].copy()

print(f"Split cutoff date: {cutoff.date()}")
print(f"Train: {len(train)} rows | abusive rate: {train[TARGET].mean():.1%}")
print(f"Test:  {len(test)} rows | abusive rate: {test[TARGET].mean():.1%}")
print()
print("Abusive rates are close between train/test -- no obvious distribution")
print("shift introduced by the chronological cut.")

Split cutoff date: 2025-10-01
Train: 7748 rows | abusive rate: 11.2%
Test:  2557 rows | abusive rate: 12.4%

Abusive rates are close between train/test -- no obvious distribution
shift introduced by the chronological cut.


## Save outputs

`train.csv`, `test.csv`, and the feature column list are what
`model/train_model.py` (notebook 03) consumes.

In [6]:
import json

train.to_csv("data/train.csv", index=False)
test.to_csv("data/test.csv", index=False)
with open("model/feature_columns.json", "w") as f:
    json.dump({
        "numeric_features": NUMERIC_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "encoded_feature_columns": feature_cols,
        "target": TARGET,
        "split_cutoff_date": str(cutoff.date()),
    }, f, indent=2)
print("Saved data/train.csv, data/test.csv, model/feature_columns.json")

Saved data/train.csv, data/test.csv, model/feature_columns.json
